In [ ]:
#NETWORK - CENTRALITY-SPACE (Readme)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Module conducts the following analysis:

# ---- Computation of network centrality scores (marker for spillover potential)
# ---- Figure 3: Spatial spillover poptential of green H2 offtakers across regions
# ---- Figure 4: Spatial spillover poptential of green H2 offtakers across sectors
# ---- Extended Data Figure 1: H2 Valleys
# ---- Extended Data Figure 3: Distribution of spillover potential excl. heating

# Module is input for:

# ---- Simulation-policy-intervention

In [ ]:
#SET-UP
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
setwd("/home/h1604190/Spatially-informed Demand-side Policies for Green H2 Diffusion/") 
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.getenv("PROJ_LIB")

check_and_load <- function(packages) {
  for (pkg in packages) {
    if (!requireNamespace(pkg, quietly = TRUE)) {
      message(paste("Installing missing package:", pkg))
      install.packages(pkg, dependencies = TRUE, repos = "https://cloud.r-project.org")
    }
    if (!(pkg %in% (.packages()))) {
      suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
  }
}

# Required libraries
required_packages <- c(
  "dplyr",          # data manipulation
  "data.table",     # fast data tables
  "tidyverse",      # tidy tools (ggplot2, dplyr, tidyr)
  "tibble",         # tibbles
  "jsonlite",       # json handling
  "tidyr",          # tidy reshaping

  "sf",             # spatial data
  "giscoR",         # gisco/eurostat shapes
  "geosphere",      # geodesic distances
  "rnaturalearth",  # natural earth basemaps
  "ggspatial",      # map annotations

  "igraph",         # network analysis
  "Matrix",         # sparse matrices

  "ggplot2",        # plotting
  "ggsci",          # color scale
  "viridis",        # viridis palettes
  "scales",         # axis scaling
  "ggridges",       # ridge plots
  "forcats",        # factor tools
  "patchwork",      # multi-panel layouts
  "ggnewscale",     # multiple scales
  "ggrepel",         # labels

  "openxlsx",        # export source data to xlsx
  "purrr"           # export source data to xlsx

)

# Load all required packages (auto-install if missing)
check_and_load(required_packages)


# Font
theme_set(
  theme_minimal(base_family = "Arial")
)

In [ ]:
#MODULE-SPECIFIC INPUTS AND SETTINGS
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Basemap
world <- ne_countries(scale = "medium", returnclass = "sf")

# EU defined H2 Valleys
valley_coords <- tibble::tibble(
  name = c("HEAVENN", "NAHV", "BalticSeaH2", "IMAGHyNE", "HI2 Valley", "CyLH2Valley"),
  lon = c(6.0, 13.5, 25.0, 4.5, 14.5, -5.0),
  lat = c(53.0, 45.5, 60.0, 45.5, 47.0, 41.8)
)
additional_valleys <- tibble::tibble(
  name = c("GreenHysland2", "TRIERES", "CRAVE-H2", "SH2AMROCK",
           "TH2ICINO", "LuxHyVal", "ZAHYR", "CONVEY", "AdvancedH2Valley",
           "H2tALENT", "HySPARK", "EASTGATEH2V"),
  lon = c(3.0, 22.9, 25.2, -9.0, 9.5, 6.1, 25.6, 9.95, -1.0, -8.0, 21.0, 21.2),
  lat = c(39.6, 37.9, 35.2, 53.3, 45.5, 49.8, 42.4, 57.6, 47.5, 38.0, 52.2, 48.7)
)

valleys_sf <- st_as_sf(valley_coords, coords=c("lon","lat"), crs=4326) %>%
  mutate(type="Large-scale H2 Valley")

additional_valleys_sf <- st_as_sf(additional_valleys, coords=c("lon","lat"), crs=4326) %>%
  mutate(type="Small-scale H2 Valley")

empty_valleys <- valleys_sf[0, ]
empty_additional <- additional_valleys_sf[0, ]

In [ ]:
#DATAFILES
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
offtakers <- readRDS("offtakers.rds")
neighbors_matrix <- readRDS("neighbors_matrix.rds")
offtakers_4326 <- st_transform(offtakers, crs = 4326)

world <- rnaturalearth::ne_countries(scale = "medium", returnclass = "sf") %>%
  st_crop(xmin = -15, xmax = 45, ymin = 30, ymax = 75) %>%
  st_transform(crs = 4326)

nuts2_shapefile <- giscoR::gisco_get_nuts(year = 2021, nuts_level = 2, resolution = "20")
if (is.na(st_crs(nuts2_shapefile))) st_crs(nuts2_shapefile) <- 4258
nuts2_shapefile <- st_transform(nuts2_shapefile, crs = 4326)

In [ ]:
#COMPUTE NETWORK CENTRALITY SCORES
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Degree Centrality
offtakers_4326$degree <- rescale(colSums(neighbors_matrix), to = c(0, 1))

# Betweenness Centrality
g <- graph_from_adjacency_matrix(neighbors_matrix, mode = "directed", weighted = NULL)
offtakers_4326$betweenness <- rescale(betweenness(g, directed = TRUE, normalized = FALSE), to = c(0, 1))

# Add to projected df
offtakers <- offtakers%>%
  left_join(
    offtakers_4326 %>%
      st_drop_geometry() %>%
      select(plant_id, degree, betweenness),
    by = "plant_id"
  )

#Save datfile
saveRDS(offtakers, "offtakers_centrality.rds")


#Reproject to long
offtakers_long <- offtakers %>%
  st_transform(4326) %>%
  select(plant_id, geometry, degree,betweenness) %>%
  tidyr::pivot_longer(
    cols = c(degree, betweenness),
    names_to = "centrality_type",
    values_to = "centrality_score"
  ) %>%
  mutate(alpha_value = ifelse(centrality_score < 0.05, 0.1, 0.5))

In [ ]:
#TOP NUTS-2 REGIONS BY SHARE OF TOP OFFTAKERS (TEXT)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Identify global top-10% offtakers for degree and betweenness centrality
nuts2_shapefile <- st_transform(nuts2_shapefile, 3035)
offtakers_nuts2 <- offtakers %>%
  st_join(nuts2_shapefile %>% select(NUTS_ID))  

# Long table of scores
cent_long <- offtakers_nuts2 %>%
  st_drop_geometry() %>%
  select(NUTS_ID, degree, betweenness) %>%
  tidyr::pivot_longer(
    cols = c(degree, betweenness),
    names_to = "metric", values_to = "score"
  )

top10_global_subset <- cent_long %>%
  filter(metric %in% c("degree", "betweenness")) %>%   # keep only these two
  group_by(metric) %>%
  mutate(
    cutoff = quantile(score, 0.9, na.rm = TRUE),
    is_top10 = score >= cutoff
  ) %>%
  ungroup()

# Count how many top offtakers per NUTS2

nuts2_top_shares_subset <- top10_global_subset %>%
  filter(is_top10) %>%                     
  group_by(metric, NUTS_ID) %>%
  summarise(n_top = n(), .groups = "drop") %>%
  group_by(metric) %>%
  mutate(
    total_top = sum(n_top),
    share_pct = 100 * n_top / total_top
  ) %>%
  ungroup() %>%
  left_join(
    st_drop_geometry(nuts2_shapefile) %>% select(NUTS_ID, NAME_LATN),
    by = "NUTS_ID"
  ) %>%
  arrange(metric, desc(share_pct))

nuts2_top_shares_subset

In [ ]:
#TOP NUTS-2 REGIONS BY SHARE OF TOP OFFTAKERS (TEXT)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Identify top-10% offtakers by metric
cent_long_subset <- cent_long %>%
  filter(metric %in% c("degree", "betweenness"))
top10_subset <- cent_long_subset %>%
  group_by(metric) %>%
  mutate(
    cutoff   = quantile(score, 0.9, na.rm = TRUE),
    is_top10 = score >= cutoff
  ) %>%
  ungroup()

# Counts and shares by NUTS2-region
nuts2_top_stats <- top10_subset %>%
  filter(is_top10) %>%                     
  group_by(metric, NUTS_ID) %>%
  summarise(n_top = n(), .groups = "drop") %>%
  group_by(metric) %>%
  mutate(
    total_top = sum(n_top),
    share_pct = 100 * n_top / total_top
  ) %>%
  ungroup()

# Add region clear names
nuts2_top_stats_named <- nuts2_top_stats %>%
  left_join(
    st_drop_geometry(nuts2_shapefile) %>% select(NUTS_ID, NAME_LATN),
    by = "NUTS_ID"
  )

# Add cumulative shares, sort descending

nuts2_top_stats_cumulative <- nuts2_top_stats_named %>%
  arrange(metric, desc(share_pct)) %>%
  group_by(metric) %>%
  mutate(
    cumulative_share_pct = cumsum(share_pct)
  ) %>%
  ungroup() %>%
  select(metric, NAME_LATN, n_top, share_pct, cumulative_share_pct)

# Print

nuts2_top_stats_cumulative


In [ ]:
# FIGURE 3: DISTRIBUTION OF SPILLOVER POTENTIAL BY REGION
# ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# centrality scale transformations
power_04_trans <- scales::trans_new(
  name = "power_04",
  transform = function(x) x^0.4,
  inverse = function(x) x^(1 / 0.4)
)

sqrt_trans <- scales::trans_new(
  name = "sqrt_custom",
  transform = function(x) sqrt(x),
  inverse = function(x) x^2
)

identity_trans <- scales::trans_new(
  name = "identity_custom",
  transform = function(x) x,
  inverse = function(x) x
)

# use identity scale for percentile values
transformation <- identity_trans

# plotting settings
excluded_sector <- "XX"

pop_max_cutoff_zoom1 <- 1000000
pop_max_cutoff_zoom2 <- 250000
highlight <- 50
common_xlim <- c(-15, 45)
common_ylim <- c(30, 75)
mean_lat_global <- mean(common_ylim)

# helper for longitude scaling in zoom windows
deg2rad <- function(d) d * pi / 180

# zoom box 1: north rhine-westphalia
zoom1_center <- c(lon = 7.0, lat = 51.5)
zoom1_height <- 2
zoom1_width <- zoom1_height *
  (diff(common_xlim) * cos(deg2rad(mean_lat_global))) /
  (diff(common_ylim) * cos(deg2rad(zoom1_center["lat"])))

zoom1_xlim <- c(
  zoom1_center["lon"] - zoom1_width / 2,
  zoom1_center["lon"] + zoom1_width / 2
)

zoom1_ylim <- c(
  zoom1_center["lat"] - zoom1_height / 2,
  zoom1_center["lat"] + zoom1_height / 2
)

# zoom box 2: eastern austria / moravia
zoom2_center <- c(lon = 16.5, lat = 48.2)
zoom2_height <- 3
zoom2_width <- zoom2_height *
  (diff(common_xlim) * cos(deg2rad(mean_lat_global))) /
  (diff(common_ylim) * cos(deg2rad(zoom2_center["lat"])))

zoom2_xlim <- c(
  zoom2_center["lon"] - zoom2_width / 2,
  zoom2_center["lon"] + zoom2_width / 2
)

zoom2_ylim <- c(
  zoom2_center["lat"] - zoom2_height / 2,
  zoom2_center["lat"] + zoom2_height / 2
)

# prepare long centrality data and convert scores to within-metric percentiles
offtakers_long <- offtakers %>%
  st_transform(4326) %>%
  filter(sector != excluded_sector) %>%
  select(
    plant_id,
    installation_name,
    geometry,
    sector,
    degree,
    betweenness
  ) %>%
  pivot_longer(
    cols = c(degree, betweenness),
    names_to = "centrality_type",
    values_to = "centrality_score"
  ) %>%
  group_by(centrality_type) %>%
  mutate(
    centrality_pct = percent_rank(centrality_score)
  ) %>%
  ungroup()

# helper to build bbox polygons
make_bbox_polygon <- function(xmin, xmax, ymin, ymax, crs = 4326) {
  coords <- matrix(
    c(
      xmin, ymin,
      xmin, ymax,
      xmax, ymax,
      xmax, ymin,
      xmin, ymin
    ),
    ncol = 2,
    byrow = TRUE
  )

  st_sfc(st_polygon(list(coords)), crs = crs)
}

bbox_zoom1 <- make_bbox_polygon(
  zoom1_xlim[1], zoom1_xlim[2],
  zoom1_ylim[1], zoom1_ylim[2]
)

bbox_zoom2 <- make_bbox_polygon(
  zoom2_xlim[1], zoom2_xlim[2],
  zoom2_ylim[1], zoom2_ylim[2]
)

# helper to extract top n sites within a zoom area
get_topN_zoom <- function(data, metric, bbox, n = highlight) {
  data %>%
    filter(centrality_type == metric) %>%
    st_intersection(bbox) %>%
    arrange(desc(centrality_score)) %>%
    slice_head(n = n)
}

# sectors represented among highlighted top sites
active_deg1 <- get_topN_zoom(
  offtakers_long,
  "degree",
  bbox_zoom1
) %>%
  st_drop_geometry() %>%
  pull(sector) %>%
  unique()

active_bet2 <- get_topN_zoom(
  offtakers_long,
  "betweenness",
  bbox_zoom2
) %>%
  st_drop_geometry() %>%
  pull(sector) %>%
  unique()

# sector palette
nrc_base10 <- pal_npg("nrc")(10)

extra_colors <- c(
  "#7A7A7A",
  "#8C564B"
)

nrc_colors <- c(nrc_base10, extra_colors)

sector_levels <- sort(unique(offtakers_long$sector))

special_colors <- c(
  "Other" = "#808080",
  "Iron & steel" = nrc_colors[1],
  "Non-metallic minerals" = nrc_colors[5]
)

special_colors <- special_colors[names(special_colors) %in% sector_levels]

reserved_colors <- unique(unname(special_colors))

regular_sector_levels <- setdiff(sector_levels, names(special_colors))

regular_colors <- setdiff(nrc_colors, reserved_colors)

if (length(regular_sector_levels) > length(regular_colors)) {
  stop("Not enough non-reserved colours for all regular sectors.")
}

sector_palette <- c(
  setNames(regular_colors[seq_along(regular_sector_levels)], regular_sector_levels),
  special_colors
)

sector_palette <- sector_palette[sector_levels]

offtakers_long$sector <- factor(
  offtakers_long$sector,
  levels = sector_levels
)

# city labels for zoom panels
cities <- ne_download(
  scale = 10,
  type = "populated_places",
  category = "cultural",
  returnclass = "sf"
)

cities_zoom1 <- st_intersection(cities, bbox_zoom1) %>%
  filter(POP_MAX > pop_max_cutoff_zoom1)

cities_zoom2 <- st_intersection(cities, bbox_zoom2) %>%
  filter(POP_MAX > pop_max_cutoff_zoom2)

# sectors that receive ring highlighting in the legend
ring_sectors_global <- union(active_deg1, active_bet2)

legend_df <- data.frame(
  sector = sector_levels,
  is_ring = sector_levels %in% ring_sectors_global,
  x = 1000,
  y = 1000
)

# europe-wide centrality map
plot_centrality_simple <- function(
  data,
  metric,
  title,
  valleys,
  valleys_small,
  show_legend = FALSE,
  bbox = NULL
) {
  ggplot() +
    geom_sf(
      data = world,
      fill = "grey97",
      color = "grey85",
      size = 0.2
    ) +
    geom_sf(
      data = filter(data, centrality_type == metric),
      aes(fill = centrality_pct),
      shape = 21,
      size = 0.7,
      stroke = 0,
      alpha = 0.7
    ) +
    geom_sf(
      data = valleys,
      aes(shape = type, color = type),
      size = 3.5,
      stroke = 1.2,
      fill = NA
    ) +
    geom_sf(
      data = valleys_small,
      aes(shape = type, color = type),
      size = 3.0,
      stroke = 1.2,
      fill = NA
    ) +
{
  if (!is.null(bbox)) list(
    geom_sf(
      data = bbox,
      fill = NA,
      color = "white",
      linewidth = 2.8
    ),
    geom_sf(
      data = bbox,
      fill = NA,
      color = "black",
      linewidth = 1.2
    )
  )
} +
    scale_fill_viridis_c(
      option = "mako",
      direction = -1,
      limits = c(0, 1),
      trans = transformation,
      name = "Centrality\n(percentile)"
    ) +
    scale_shape_manual(
      values = c(
        "Large-scale H2 Valley" = 4,
        "Small-scale H2 Valley" = 1
      ),
      name = "Hydrogen valleys"
    ) +
    scale_color_manual(
      values = c(
        "Large-scale H2 Valley" = "maroon",
        "Small-scale H2 Valley" = "maroon"
      ),
      name = "Hydrogen valleys"
    ) +
    coord_sf(
      xlim = common_xlim,
      ylim = common_ylim,
      expand = FALSE
    ) +
    labs(title = title) +
    theme_classic(base_size = 16) +
    theme(
      plot.title = element_text(hjust = 0),
      legend.position = if (show_legend) "right" else "none",
      panel.border = element_rect(
        color = "black",
        fill = NA,
        linewidth = 0.6
      )
    )
}

# zoomed map with top sites ringed by sector
plot_centrality_sector <- function(
  data,
  metric,
  title,
  valleys,
  valleys_small,
  show_legend = FALSE,
  zoom = FALSE,
  zoom_xlim = NULL,
  zoom_ylim = NULL,
  cities_zoom = NULL
) {
  data_sub <- data %>%
    filter(centrality_type == metric)

  topN <- get_topN_zoom(
    data,
    metric,
    make_bbox_polygon(
      zoom_xlim[1],
      zoom_xlim[2],
      zoom_ylim[1],
      zoom_ylim[2]
    )
  )

  data_sub <- data_sub %>%
    mutate(is_topN = plant_id %in% topN$plant_id)

  ggplot() +
    geom_sf(
      data = world,
      fill = "grey97",
      color = "grey85",
      size = 0.2
    ) +
    geom_sf(
      data = data_sub %>% filter(!is_topN),
      aes(fill = centrality_pct),
      shape = 21,
      size = 2,
      stroke = 0,
      alpha = 0.4
    ) +
    geom_sf(
      data = topN,
      aes(fill = centrality_pct),
      shape = 21,
      size = 7,
      stroke = 0,
      alpha = 0.3
    ) +
    geom_sf(
      data = topN,
      aes(color = sector),
      shape = 21,
      size = 7,
      stroke = 2,
      fill = NA
    ) +
    # invisible points used to force full legend entries
    geom_point(
      data = legend_df,
      aes(x = x, y = y, color = sector),
      shape = 16,
      size = 3,
      inherit.aes = FALSE
    ) +
    geom_point(
      data = legend_df %>% filter(is_ring),
      aes(x = x, y = y, color = sector),
      shape = 21,
      fill = NA,
      size = 7,
      stroke = 2,
      inherit.aes = FALSE
    ) +
    {
      if (zoom) geom_sf(
        data = cities_zoom,
        color = "black",
        size = 1.2
      )
    } +
    {
      if (zoom) geom_label_repel(
        data = cities_zoom,
        aes(label = NAME, geometry = geometry),
        stat = "sf_coordinates",
        size = 4,
        fill = "white",
        box.padding = 0.4,
        label.size = 0.2
      )
    } +
    scale_fill_viridis_c(
      option = "mako",
      direction = -1,
      limits = c(0, 1),
      trans = transformation,
      name = "Centrality\n(percentile)"
    ) +
    scale_color_manual(
      name = "Top 50 offtakers / Sector",
      values = sector_palette,
      breaks = sector_levels,
      drop = FALSE,
      guide = if (show_legend) "legend" else "none"
    ) +
    coord_sf(
      xlim = zoom_xlim,
      ylim = zoom_ylim,
      expand = FALSE
    ) +
    labs(title = title) +
    theme_classic(base_size = 16) +
    theme(
      plot.title = element_text(hjust = 0),
      legend.position = if (show_legend) "right" else "none",
      panel.border = element_rect(
        color = "black",
        fill = NA,
        linewidth = 0.6
      )
    )
}

# panel a
p1 <- plot_centrality_simple(
  offtakers_long,
  "degree",
  "H₂ Valleys | Degree Centrality",
  valleys_sf,
  additional_valleys_sf,
  show_legend = FALSE,
  bbox = bbox_zoom1
)

# panel b
p2 <- plot_centrality_simple(
  offtakers_long,
  "betweenness",
  "H₂ Corridors | Betweenness Centrality",
  empty_valleys,
  empty_additional,
  show_legend = TRUE,
  bbox = bbox_zoom2
)

# panel c
p3 <- plot_centrality_sector(
  offtakers_long,
  "degree",
  "North Rhine-Westphalia",
  valleys_sf,
  additional_valleys_sf,
  zoom = TRUE,
  show_legend = FALSE,
  zoom_xlim = zoom1_xlim,
  zoom_ylim = zoom1_ylim,
  cities_zoom = cities_zoom1
)

# panel d
p4 <- plot_centrality_sector(
  offtakers_long,
  "betweenness",
  "Eastern Austria / Moravia",
  valleys_sf,
  additional_valleys_sf,
  zoom = TRUE,
  show_legend = TRUE,
  zoom_xlim = zoom2_xlim,
  zoom_ylim = zoom2_ylim,
  cities_zoom = cities_zoom2
)

# combine panels
figure3 <- ((p1 | p2) / (p3 | p4)) +
  plot_annotation(
    tag_levels = "a",
    theme = theme(
      plot.tag = element_text(
        face = "bold",
        size = 14
      )
    )
  ) +
  plot_layout(
    guides = "collect",
    widths = c(1, 1),
    heights = c(1, 1)
  ) &
  theme(
    legend.position = "right",
    panel.spacing = unit(0, "cm"),
    axis.text = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank(),
    plot.margin = margin(0, 0, 0, 0)
  )

# display settings
options(
  repr.plot.width = 16,
  repr.plot.height = 12,
  repr.plot.res = 600
)

print(figure3)

# export figure
ggsave(
  "figure3.png",
  figure3,
  width = 16,
  height = 12,
  units = "in",
  dpi = 600,
  limitsize = FALSE
)

In [ ]:
# EXPORT FIGURE 3 SOURCE DATA TO XLSX

# Folder / filename
out_file <- "figure3_source_data.xlsx"


# Prepare installation-level map data
offtakers_source <- offtakers_long %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2],
    centrality_type = as.character(centrality_type),
    sector = as.character(sector)
  ) %>%
  st_drop_geometry() %>%
  select(
    plant_id,
    installation_name,
    sector,
    centrality_type,
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  ) %>%
  arrange(
    centrality_type,
    desc(centrality_score),
    plant_id
  )


# Prepare panel-specific centrality data
panel_a_degree <- offtakers_source %>%
  filter(centrality_type == "degree")

panel_b_betweenness <- offtakers_source %>%
  filter(centrality_type == "betweenness")


# Reproduce highlighted top-50 sites in zoom panels

panel_c_top50_degree <- get_topN_zoom(
  data = offtakers_long,
  metric = "degree",
  bbox = bbox_zoom1,
  n = highlight
) %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    rank_within_zoom = row_number(),
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  )

panel_d_top50_betweenness <- get_topN_zoom(
  data = offtakers_long,
  metric = "betweenness",
  bbox = bbox_zoom2,
  n = highlight
) %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    rank_within_zoom = row_number(),
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  )


# Prepare all sites shown inside each zoom panel

panel_c_all_sites <- offtakers_long %>%
  filter(centrality_type == "degree") %>%
  st_filter(bbox_zoom1, .predicate = st_intersects) %>%
  mutate(
    is_top50 = plant_id %in% panel_c_top50_degree$plant_id,
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    is_top50,
    longitude,
    latitude
  ) %>%
  arrange(
    desc(is_top50),
    desc(centrality_score),
    plant_id
  )

panel_d_all_sites <- offtakers_long %>%
  filter(centrality_type == "betweenness") %>%
  st_filter(bbox_zoom2, .predicate = st_intersects) %>%
  mutate(
    is_top50 = plant_id %in% panel_d_top50_betweenness$plant_id,
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    is_top50,
    longitude,
    latitude
  ) %>%
  arrange(
    desc(is_top50),
    desc(centrality_score),
    plant_id
  )

# Prepare hydrogen-valley point data

prepare_valley_source <- function(x, source_group) {

  if (nrow(x) == 0) {
    return(
      tibble(
        source_group = character(),
        type = character(),
        longitude = numeric(),
        latitude = numeric()
      )
    )
  }

  x %>%
    st_transform(4326) %>%
    mutate(
      longitude = st_coordinates(.)[, 1],
      latitude = st_coordinates(.)[, 2],
      source_group = source_group
    ) %>%
    st_drop_geometry() %>%
    relocate(
      source_group,
      type,
      longitude,
      latitude
    )
}

hydrogen_valleys_source <- bind_rows(
  prepare_valley_source(
    valleys_sf,
    "Main hydrogen-valley dataset"
  ),
  prepare_valley_source(
    additional_valleys_sf,
    "Additional hydrogen-valley dataset"
  )
)

# Prepare city-label data

prepare_city_source <- function(x, panel_name) {

  if (nrow(x) == 0) {
    return(
      tibble(
        panel = character(),
        city = character(),
        population_max = numeric(),
        longitude = numeric(),
        latitude = numeric()
      )
    )
  }

  x %>%
    st_transform(4326) %>%
    mutate(
      longitude = st_coordinates(.)[, 1],
      latitude = st_coordinates(.)[, 2]
    ) %>%
    st_drop_geometry() %>%
    transmute(
      panel = panel_name,
      city = NAME,
      population_max = POP_MAX,
      longitude,
      latitude
    ) %>%
    arrange(desc(population_max), city)
}

city_labels_source <- bind_rows(
  prepare_city_source(
    cities_zoom1,
    "Panel c: North Rhine-Westphalia"
  ),
  prepare_city_source(
    cities_zoom2,
    "Panel d: Eastern Austria / Moravia"
  )
)

# Prepare map extents and zoom-box coordinates

map_extents <- tibble(
  panel = c(
    "Panels a-b: Europe",
    "Panel c: North Rhine-Westphalia",
    "Panel d: Eastern Austria / Moravia"
  ),
  xmin = c(
    common_xlim[1],
    zoom1_xlim[1],
    zoom2_xlim[1]
  ),
  xmax = c(
    common_xlim[2],
    zoom1_xlim[2],
    zoom2_xlim[2]
  ),
  ymin = c(
    common_ylim[1],
    zoom1_ylim[1],
    zoom2_ylim[1]
  ),
  ymax = c(
    common_ylim[2],
    zoom1_ylim[2],
    zoom2_ylim[2]
  ),
  coordinate_reference_system = "EPSG:4326"
)

zoom_box_vertices <- bind_rows(
  tibble(
    panel = "Panel a zoom box",
    vertex_order = 1:5,
    longitude = c(
      zoom1_xlim[1],
      zoom1_xlim[1],
      zoom1_xlim[2],
      zoom1_xlim[2],
      zoom1_xlim[1]
    ),
    latitude = c(
      zoom1_ylim[1],
      zoom1_ylim[2],
      zoom1_ylim[2],
      zoom1_ylim[1],
      zoom1_ylim[1]
    )
  ),
  tibble(
    panel = "Panel b zoom box",
    vertex_order = 1:5,
    longitude = c(
      zoom2_xlim[1],
      zoom2_xlim[1],
      zoom2_xlim[2],
      zoom2_xlim[2],
      zoom2_xlim[1]
    ),
    latitude = c(
      zoom2_ylim[1],
      zoom2_ylim[2],
      zoom2_ylim[2],
      zoom2_ylim[1],
      zoom2_ylim[1]
    )
  )
)

# Prepare sector palette and legend information

sector_palette_source <- tibble(
  sector = names(sector_palette),
  colour_hex = unname(sector_palette),
  represented_among_highlighted_sites =
    names(sector_palette) %in% ring_sectors_global
)


# Figure metadata

figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Coordinate reference system",
    "Excluded sector",
    "Centrality metrics",
    "Centrality display unit",
    "Centrality percentile calculation",
    "Centrality colour scale",
    "Centrality scale transformation",
    "Number of highlighted sites per zoom panel",
    "City population threshold",
    "Europe-wide longitude range",
    "Europe-wide latitude range",
    "Panel c centre longitude",
    "Panel c centre latitude",
    "Panel d centre longitude",
    "Panel d centre latitude"
  ),
  value = c(
    "Figure 3: Distribution of spillover potential by region",
    paste(
      "Europe-wide degree and betweenness centrality maps,",
      "with zoom panels showing the 50 highest-ranked offtakers",
      "within North Rhine-Westphalia and Eastern Austria / Moravia"
    ),
    "EPSG:4326",
    excluded_sector,
    "Degree centrality; Betweenness centrality",
    "Within-metric percentile, ranging from 0 to 1",
    paste(
      "percent_rank applied separately within degree and",
      "betweenness centrality"
    ),
    "Viridis mako, reversed",
    "Identity",
    as.character(highlight),
    as.character(pop_max_cutoff),
    paste(common_xlim, collapse = " to "),
    paste(common_ylim, collapse = " to "),
    as.character(zoom1_center["lon"]),
    as.character(zoom1_center["lat"]),
    as.character(zoom2_center["lon"]),
    as.character(zoom2_center["lat"])
  )
)

# Panel descriptions

panel_metadata <- tibble(
  panel = c(
    "a",
    "b",
    "c",
    "d"
  ),
  title = c(
    "H2 Valleys | Degree Centrality",
    "H2 Corridors | Betweenness Centrality",
    "North Rhine-Westphalia",
    "Eastern Austria / Moravia"
  ),
  centrality_metric = c(
    "degree",
    "betweenness",
    "degree",
    "betweenness"
  ),
  geographic_extent = c(
    "Europe",
    "Europe",
    "North Rhine-Westphalia zoom window",
    "Eastern Austria / Moravia zoom window"
  ),
  highlighted_sites = c(
    NA_integer_,
    NA_integer_,
    highlight,
    highlight
  )
)


# Source sheets

source_sheets <- list(
  "README" = figure_metadata,

  "Panel_descriptions" = panel_metadata,

  "All_offtakers" = offtakers_source,

  "Panel_a_degree" = panel_a_degree,

  "Panel_b_betweenness" = panel_b_betweenness,

  "Panel_c_all_sites" = panel_c_all_sites,

  "Panel_c_top50" = panel_c_top50_degree,

  "Panel_d_all_sites" = panel_d_all_sites,

  "Panel_d_top50" = panel_d_top50_betweenness,

  "Hydrogen_valleys" = hydrogen_valleys_source,

  "City_labels" = city_labels_source,

  "Map_extents" = map_extents,

  "Zoom_box_vertices" = zoom_box_vertices,

  "Sector_palette" = sector_palette_source
)


# Write workbook

wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  sheet_data <- source_sheets[[sheet_name]]

  addWorksheet(
    wb,
    sheetName = sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = sheet_data
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  if (ncol(sheet_data) > 0) {
    addFilter(
      wb,
      sheet = sheet_name,
      row = 1,
      cols = seq_len(ncol(sheet_data))
    )

    setColWidths(
      wb,
      sheet = sheet_name,
      cols = seq_len(ncol(sheet_data)),
      widths = "auto"
    )
  }
})

saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)

In [ ]:
#FIGURE 4: DISTRIBUTION OF SPILLOVER POTENTIAL BY SECTOR
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# shared plot theme
style_template <- theme_minimal(base_size = 22) +
  theme(
    axis.line          = element_line(color = "black"),
    axis.ticks         = element_line(color = "black"),
    axis.text          = element_text(size = 22, color = "black"),
    axis.title         = element_text(size = 22, color = "black"),
    plot.title         = element_text(size = 22, hjust = 0.5, color = "black"),
    axis.text.x.top    = element_blank(),
    axis.text.y.right  = element_blank(),
    axis.ticks.x.top   = element_blank(),
    axis.ticks.y.right = element_blank(),
    panel.grid.minor   = element_blank(),
    panel.grid.major   = element_blank(),
    legend.text        = element_text(size = 22),
    legend.title       = element_text(size = 22)
  )

# reshape centrality measures to long format
centrality_long <- offtakers_4326 %>%
  st_drop_geometry() %>%
  select(
    sector,
    Degree = degree,
    Betweenness = betweenness
  ) %>%
  pivot_longer(
    -sector,
    names_to = "centrality_type",
    values_to = "centrality_value"
  ) %>%
  mutate(
    centrality_type = factor(
      centrality_type,
      levels = c("Degree", "Betweenness")
    )
  )

# sector-level counts and sector grouping
sector_stats <- offtakers_4326 %>%
  st_drop_geometry() %>%
  group_by(sector) %>%
  summarise(
    n_offtakers = n(),
    median_outdeg = median(degree, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    group = case_when(
      sector %in% c("Aviation", "Shipping", "Heavy duty") ~ "Transport",
      sector == "Heat" ~ "Heat",
      sector == "Power" ~ "Power",
      TRUE ~ "Industrials"
    ),
    group = factor(group, levels = c("Industrials", "Transport", "Heat", "Power"))
  )

# order sectors for plotting
industrials <- sector_stats %>%
  filter(group == "Industrials") %>%
  arrange(sector) %>%
  pull(sector)

transport <- sector_stats %>%
  filter(group == "Transport") %>%
  arrange(match(sector, c("Aviation", "Shipping", "Heavy duty"))) %>%
  pull(sector)

heat <- sector_stats %>%
  filter(group == "Heat") %>%
  arrange(sector) %>%
  pull(sector)

power <- sector_stats %>%
  filter(group == "Power") %>%
  arrange(sector) %>%
  pull(sector)

sector_order <- c(industrials, transport, heat, power)

# apply sector order to both datasets
sector_stats <- sector_stats %>%
  mutate(y_axis_label = factor(sector, levels = rev(sector_order)))

centrality_long <- centrality_long %>%
  left_join(sector_stats %>% select(sector, group), by = "sector") %>%
  mutate(y_axis_label = factor(sector, levels = rev(sector_order)))

# calculate 90th percentile threshold by centrality type
percentiles_df <- centrality_long %>%
  group_by(centrality_type) %>%
  summarise(
    p90 = quantile(centrality_value, 0.9, na.rm = TRUE),
    .groups = "drop"
  )

# compute sectoral share above threshold
above_threshold <- centrality_long %>%
  left_join(percentiles_df, by = "centrality_type") %>%
  group_by(y_axis_label, centrality_type) %>%
  summarise(
    share_above = mean(centrality_value > p90, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    x_pos = 1.12,
    label = scales::percent(share_above, accuracy = 1)
  )

# split industrial vs non-industrial sectors for ridge styling
centrality_industrial <- centrality_long %>%
  filter(group == "Industrials")

centrality_other <- centrality_long %>%
  filter(group != "Industrials")

# shared y-axis settings
y_levels <- levels(sector_stats$y_axis_label)
y_scale <- scale_y_discrete(
  limits = y_levels,
  expand = expansion(mult = c(0.01, 0.05))
)
ylims_cart <- c(0.5, length(y_levels) + 0.5)

# bar plot of sector counts
xmax_val <- 8000

bar_plot <- ggplot(
  sector_stats,
  aes(x = n_offtakers, y = y_axis_label, fill = group)
) +
  geom_col(width = 0.7) +
  geom_vline(xintercept = xmax_val, color = "black", linewidth = 0.6) +
  scale_fill_manual(
    values = c(
      "Industrials" = "#4DBBD5FF",
      "Transport" = "#6B6B6B",
      "Heat" = "#B0B0B0",
      "Power" = "#E6E6E6"
    ),
    name = "Sector group"
  ) +
  labs(x = "", y = NULL, title = "# Offtakers") +
  scale_x_continuous(
    breaks = scales::pretty_breaks(),
    labels = scales::label_number(accuracy = 1),
    sec.axis = dup_axis(name = NULL, labels = NULL),
    expand = c(0, 0)
  ) +
  y_scale +
  coord_cartesian(
    xlim = c(0, xmax_val),
    ylim = ylims_cart,
    clip = "off"
  ) +
  style_template +
  theme(
    axis.line.y.right  = element_line(color = "black"),
    axis.ticks.y.right = element_blank(),
    axis.text.y.right  = element_blank(),
    axis.text.y        = element_text(hjust = 0, size = 22, color = "black")
  )

# ridge plot function for one centrality type
ridge_plot <- function(type, title, xlab) {
  p90_val <- filter(percentiles_df, centrality_type == type)$p90

  ggplot() +
    geom_rect(
      aes(xmin = p90_val, xmax = 1.2, ymin = -Inf, ymax = Inf),
      inherit.aes = FALSE,
      fill = "#EAE6F4",
      alpha = 0.4
    ) +
    geom_density_ridges_gradient(
      data = filter(centrality_other, centrality_type == type),
      aes(x = centrality_value, y = y_axis_label, fill = ..x..),
      scale = 0.8,
      trim = TRUE,
      rel_min_height = 0.01,
      gradient_lwd = 0.3,
      color = "black"
    ) +
    scale_fill_gradient(low = "grey90", high = "grey20", guide = "none") +
    new_scale_fill() +
    geom_density_ridges_gradient(
      data = filter(centrality_industrial, centrality_type == type),
      aes(x = centrality_value, y = y_axis_label, fill = ..x..),
      scale = 0.8,
      trim = TRUE,
      rel_min_height = 0.01,
      gradient_lwd = 0.3,
      color = "black"
    ) +
    scale_fill_viridis_c(
      name = "Centrality",
      option = "mako",
      direction = -1,
      end = 0.9
    ) +
    geom_point(
      data = filter(above_threshold, centrality_type == type),
      aes(
        x = x_pos,
        y = y_axis_label,
        shape = "Share above\n90th percentile",
        stroke = share_above * 5
      ),
      inherit.aes = FALSE,
      size = 16,
      fill = "white",
      color = "black"
    ) +
    geom_text(
      data = filter(above_threshold, centrality_type == type),
      aes(x = x_pos, y = y_axis_label, label = label),
      inherit.aes = FALSE,
      size = 5.6,
      color = "black",
      hjust = 0.5,
      vjust = 0.5
    ) +
    geom_vline(
      data = filter(percentiles_df, centrality_type == type),
      aes(xintercept = p90, linetype = "90% percentile"),
      color = "black",
      linewidth = 0.6
    ) +
    geom_vline(xintercept = 1.2, color = "black", linewidth = 0.6) +
    scale_x_continuous(
      limits = c(0, 1.2),
      expand = c(0, 0),
      sec.axis = dup_axis(name = NULL, labels = NULL),
      labels = function(x) ifelse(x == 0, "0", as.character(x))
    ) +
    y_scale +
    scale_shape_manual("", values = c("Share above\n90th percentile" = 21)) +
    scale_linetype_manual("", values = c("90% percentile" = "solid")) +
    labs(x = xlab, y = NULL, title = title) +
    coord_cartesian(ylim = ylims_cart, clip = "off") +
    style_template +
    theme(
      axis.text.y = element_blank(),
      axis.ticks.y = element_blank(),
      axis.title.y = element_blank(),
      axis.line.y.right = element_line(color = "black")
    )
}

# build ridge plots
ridge_degree <- ridge_plot(
  "Degree",
  "H₂ valleys | Degree Centrality",
  "Degree centrality"
)

ridge_between <- ridge_plot(
  "Betweenness",
  "H₂ corridors | Betweenness Centrality",
  "Betweenness centrality"
)

# combine panels
figure4 <- bar_plot + ridge_degree + ridge_between +
  plot_annotation(
    tag_levels = "a",
    theme = theme(
      plot.tag = element_text(face = "bold", size = 18),
      plot.tag.position = c(0, 1)
    )
  ) +
  plot_layout(widths = c(0.5, 1, 1), guides = "collect")

# display settings
options(repr.plot.width = 30, repr.plot.height = 10, repr.plot.res = 600)

figure4

# export figure
ggsave(
  "figure4.pdf",
  figure4,
  device = cairo_pdf,
  width = 25,
  height = 12,
  units = "in",
  dpi = 600
)

In [ ]:
# EXPORT FIGURE 4 SOURCE DATA TO XLSX

# Folder / filename
out_file <- "figure4_source_data.xlsx"

# Prepare installation-level centrality data

centrality_source <- offtakers_4326 %>%
  st_drop_geometry() %>%
  select(
    plant_id,
    installation_name,
    sector,
    degree,
    betweenness
  ) %>%
  pivot_longer(
    cols = c(degree, betweenness),
    names_to = "centrality_type",
    values_to = "centrality_value"
  ) %>%
  mutate(
    centrality_type = recode(
      centrality_type,
      degree = "Degree",
      betweenness = "Betweenness"
    )
  ) %>%
  left_join(
    sector_stats %>%
      select(
        sector,
        group,
        n_offtakers,
        median_outdeg,
        y_axis_label
      ),
    by = "sector"
  ) %>%
  mutate(
    centrality_type = as.character(centrality_type),
    group = as.character(group),
    sector_plot_order = match(
      sector,
      sector_order
    )
  ) %>%
  select(
    plant_id,
    installation_name,
    sector,
    sector_group = group,
    sector_plot_order,
    centrality_type,
    centrality_value
  ) %>%
  arrange(
    centrality_type,
    sector_plot_order,
    desc(centrality_value),
    plant_id
  )

# Prepare sector-level count data for panel a

panel_a_sector_counts <- sector_stats %>%
  mutate(
    sector = as.character(sector),
    sector_group = as.character(group),
    sector_plot_order = match(
      sector,
      sector_order
    )
  ) %>%
  select(
    sector,
    sector_group,
    sector_plot_order,
    n_offtakers,
    median_degree_centrality = median_outdeg
  ) %>%
  arrange(sector_plot_order)

# Prepare ridge-distribution source data

panel_b_degree_distribution <- centrality_source %>%
  filter(centrality_type == "Degree") %>%
  select(
    plant_id,
    installation_name,
    sector,
    sector_group,
    sector_plot_order,
    degree_centrality = centrality_value
  )

panel_c_betweenness_distribution <- centrality_source %>%
  filter(centrality_type == "Betweenness") %>%
  select(
    plant_id,
    installation_name,
    sector,
    sector_group,
    sector_plot_order,
    betweenness_centrality = centrality_value
  )

# Prepare percentile thresholds

centrality_thresholds <- percentiles_df %>%
  mutate(
    centrality_type = as.character(centrality_type),
    percentile = 0.90,
    percentile_label = "90th percentile"
  ) %>%
  transmute(
    centrality_type,
    percentile,
    percentile_label,
    threshold_value = p90
  )

# Prepare sectoral shares above the 90th percentile

sector_shares_above_threshold <- above_threshold %>%
  mutate(
    sector = as.character(y_axis_label),
    centrality_type = as.character(centrality_type)
  ) %>%
  left_join(
    sector_stats %>%
      transmute(
        sector = as.character(sector),
        sector_group = as.character(group),
        n_offtakers,
        sector_plot_order = match(
          sector,
          sector_order
        )
      ),
    by = "sector"
  ) %>%
  transmute(
    sector,
    sector_group,
    sector_plot_order,
    centrality_type,
    n_offtakers,
    share_above_90th_percentile = share_above,
    percentage_above_90th_percentile = share_above * 100,
    plotted_label = label
  ) %>%
  arrange(
    centrality_type,
    sector_plot_order
  )


# Add counts above the threshold

sector_threshold_counts <- centrality_long %>%
  left_join(
    percentiles_df,
    by = "centrality_type"
  ) %>%
  group_by(
    sector,
    centrality_type
  ) %>%
  summarise(
    n_non_missing = sum(!is.na(centrality_value)),
    n_above_90th_percentile = sum(
      centrality_value > p90,
      na.rm = TRUE
    ),
    share_above_90th_percentile = mean(
      centrality_value > p90,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  left_join(
    sector_stats %>%
      transmute(
        sector,
        sector_group = as.character(group),
        sector_plot_order = match(
          sector,
          sector_order
        )
      ),
    by = "sector"
  ) %>%
  mutate(
    centrality_type = as.character(centrality_type),
    percentage_above_90th_percentile =
      share_above_90th_percentile * 100
  ) %>%
  select(
    sector,
    sector_group,
    sector_plot_order,
    centrality_type,
    n_non_missing,
    n_above_90th_percentile,
    share_above_90th_percentile,
    percentage_above_90th_percentile
  ) %>%
  arrange(
    centrality_type,
    sector_plot_order
  )


# Prepare sector-order information

sector_order_source <- tibble(
  sector = sector_order,
  sector_plot_order = seq_along(sector_order)
) %>%
  left_join(
    sector_stats %>%
      transmute(
        sector,
        sector_group = as.character(group)
      ),
    by = "sector"
  ) %>%
  select(
    sector_plot_order,
    sector,
    sector_group
  )

# Prepare sector-group definitions and colours

sector_group_source <- tibble(
  sector_group = c(
    "Industrials",
    "Transport",
    "Heat",
    "Power"
  ),
  definition = c(
    "All industrial sectors not classified as transport, heat or power",
    "Aviation, Shipping and Heavy duty",
    "Heat",
    "Power"
  ),
  fill_colour = c(
    "#4DBBD5FF",
    "#6B6B6B",
    "#B0B0B0",
    "#E6E6E6"
  )
)

# Figure metadata

figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Panel a",
    "Panel b",
    "Panel c",
    "Centrality measures",
    "Centrality value range shown",
    "Threshold",
    "Threshold comparison",
    "Share calculation",
    "Ridge density estimation",
    "Industrial ridge colour scale",
    "Non-industrial ridge colour scale",
    "Maximum x-axis value in panel a",
    "Number of sectors",
    "Number of installations",
    "Missing values"
  ),
  value = c(
    "Figure 4: Distribution of spillover potential by sector",
    paste(
      "Sector-level number of offtakers and installation-level",
      "distributions of degree and betweenness centrality"
    ),
    "Number of offtakers by sector",
    "Degree-centrality distribution by sector",
    "Betweenness-centrality distribution by sector",
    "Degree centrality; Betweenness centrality",
    "0 to 1.2 in panels b and c",
    "Centrality-specific 90th percentile across all installations",
    "centrality_value > p90",
    paste(
      "Mean of the indicator centrality_value > p90 within",
      "each sector and centrality type"
    ),
    paste(
      "Kernel-density ridges generated by",
      "geom_density_ridges_gradient"
    ),
    "Viridis mako, reversed",
    "Grey gradient",
    as.character(xmax_val),
    as.character(length(sector_order)),
    as.character(nrow(offtakers_4326)),
    "Missing centrality values excluded from medians, quantiles and shares"
  )
)

# Panel descriptions

panel_metadata <- tibble(
  panel = c(
    "a",
    "b",
    "c"
  ),
  title = c(
    "# Offtakers",
    "H2 valleys | Degree Centrality",
    "H2 corridors | Betweenness Centrality"
  ),
  source_object = c(
    "sector_stats",
    "centrality_long",
    "centrality_long"
  ),
  plotted_value = c(
    "n_offtakers",
    "Degree centrality",
    "Betweenness centrality"
  ),
  additional_annotation = c(
    NA_character_,
    "Sector share above the overall degree-centrality 90th percentile",
    "Sector share above the overall betweenness-centrality 90th percentile"
  )
)


# Source sheets

source_sheets <- list(
  "README" = figure_metadata,

  "Panel_descriptions" = panel_metadata,

  "Panel_a_sector_counts" = panel_a_sector_counts,

  "Panel_b_degree_values" = panel_b_degree_distribution,

  "Panel_c_betweenness" = panel_c_betweenness_distribution,

  "All_centrality_values" = centrality_source,

  "Centrality_thresholds" = centrality_thresholds,

  "Sector_threshold_shares" = sector_shares_above_threshold,

  "Sector_threshold_counts" = sector_threshold_counts,

  "Sector_plot_order" = sector_order_source,

  "Sector_group_definitions" = sector_group_source
)


# Write workbook

wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  sheet_data <- source_sheets[[sheet_name]]

  addWorksheet(
    wb,
    sheetName = sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = sheet_data
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  addFilter(
    wb,
    sheet = sheet_name,
    row = 1,
    cols = seq_len(ncol(sheet_data))
  )

  setColWidths(
    wb,
    sheet = sheet_name,
    cols = seq_len(ncol(sheet_data)),
    widths = "auto"
  )
})

saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)

In [ ]:
# EXTENDED DATA FIGURE 1: H2 VALLEYS MAP + EMPTY H2 CORRIDORS MAP
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

common_xlim <- c(-15, 45)
common_ylim <- c(30, 75)

# EU defined H2 Valleys
valley_coords <- tibble::tibble(
  name = c("HEAVENN", "NAHV", "BalticSeaH2", "IMAGHyNE", "HI2 Valley", "CyLH2Valley"),
  lon = c(6.0, 13.5, 25.0, 4.5, 14.5, -5.0),
  lat = c(53.0, 45.5, 60.0, 45.5, 47.0, 41.8)
)

additional_valleys <- tibble::tibble(
  name = c(
    "GreenHysland2", "TRIERES", "CRAVE-H2", "SH2AMROCK",
    "TH2ICINO", "LuxHyVal", "ZAHYR", "CONVEY", "AdvancedH2Valley",
    "H2tALENT", "HySPARK", "EASTGATEH2V"
  ),
  lon = c(3.0, 22.9, 25.2, -9.0, 9.5, 6.1, 25.6, 9.95, -1.0, -8.0, 21.0, 21.2),
  lat = c(39.6, 37.9, 35.2, 53.3, 45.5, 49.8, 42.4, 57.6, 47.5, 38.0, 52.2, 48.7)
)

# metadata from the shared image + production data
valley_meta <- tibble::tribble(
  ~name,               ~label_name,                           ~call,        ~type,                      ~h2_production_t_year,
  "HEAVENN",           "HEAVENN",                             "Call 2019",  "Large-scale H2 Valley",   36500,
  "NAHV",              "NAHV",                                "Call 2022",  "Large-scale H2 Valley",   5000,
  "BalticSeaH2",       "BalticSeaH2",                         "Call 2022",  "Large-scale H2 Valley",   150000,
  "IMAGHyNE",          "IMAGHyNE",                            "Call 2023",  "Large-scale H2 Valley",   8000,
  "HI2 Valley",        "HI2 Valley",                          "Call 2024",  "Large-scale H2 Valley",   2030,
  "CyLH2Valley",       "CyLH2Valley",                         "Call 2024",  "Large-scale H2 Valley",   16800,
  "GreenHysland2",     "Green Hysland",                       "Call 2023",  "Small-scale H2 Valley",   300,
  "TRIERES",           "TRIERES",                             "Call 2020",  "Small-scale H2 Valley",   4500,
  "CRAVE-H2",          "CRAVE-H2",                            "Call 2020",  "Small-scale H2 Valley",   500,
  "SH2AMROCK",         "SH2AMROCK",                           "Call 2020",  "Small-scale H2 Valley",   500,
  "TH2ICINO",          "TH2ICINO",                            "Call 2020",  "Small-scale H2 Valley",   2028,
  "LuxHyVal",          "LuxHyVal",                            "Call 2020",  "Small-scale H2 Valley",   650,
  "ZAHYR",             "ZAHYR",                               "Call 2020",  "Small-scale H2 Valley",   500,
  "CONVEY",            "CONVEY",                              "Call 2024",  "Small-scale H2 Valley",   500,
  "AdvancedH2Valley",  "AdvancedH2Valley",                    "Call 2024",  "Small-scale H2 Valley",   1175,
  "H2tALENT",          "H2TALENT",                            "Call 2024",  "Small-scale H2 Valley",   760,
  "HySPARK",           "HySPARK",                             "Call 2024",  "Small-scale H2 Valley",   5000,
  "EASTGATEH2V",       "EASTGATEH2V",                         "Call 2024",  "Small-scale H2 Valley",   530
)

# combine + attach metadata
valleys_all <- bind_rows(
  valley_coords %>% mutate(type = "Large-scale H2 Valley"),
  additional_valleys %>% mutate(type = "Small-scale H2 Valley")
) %>%
  left_join(valley_meta, by = c("name", "type")) %>%
  mutate(
    label_name = dplyr::coalesce(label_name, name),
    call = factor(call, levels = sort(unique(call)))
  ) %>%
  st_as_sf(coords = c("lon", "lat"), crs = 4326)

# nrc palette
n_calls <- length(levels(valleys_all$call))

nrc_colors <- pal_npg("nrc")(10)

# swap 5th and 9th colors
tmp <- nrc_colors[5]
nrc_colors[5] <- nrc_colors[9]
nrc_colors[9] <- tmp

call_palette <- setNames(
  nrc_colors[1:n_calls],
  levels(valleys_all$call)
)
# theme
# combine + attach metadata
valleys_all <- bind_rows(
  valley_coords %>% mutate(type = "Large-scale H2 Valley"),
  additional_valleys %>% mutate(type = "Small-scale H2 Valley")
) %>%
  left_join(valley_meta, by = c("name", "type")) %>%
  mutate(
    label_name = dplyr::coalesce(label_name, name),
    call = factor(call, levels = sort(unique(call)))
  ) %>%
  sf::st_as_sf(coords = c("lon", "lat"), crs = 4326)

# nrc palette with 5th and 9th swapped
nrc_colors <- pal_npg("nrc")(10)
tmp <- nrc_colors[5]
nrc_colors[5] <- nrc_colors[9]
nrc_colors[9] <- tmp

call_palette <- setNames(
  nrc_colors[1:length(levels(valleys_all$call))],
  levels(valleys_all$call)
)

# theme
theme_map <- theme_classic(base_size = 22) +
  theme(
    plot.title = element_text(size = 24, hjust = 0),
    legend.title = element_text(size = 18),
    legend.text = element_text(size = 16),
    panel.border = element_rect(color = "black", fill = NA, linewidth = 0.8),
    axis.text = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank(),
    plot.margin = margin(5, 5, 5, 5)
  )

# left: filled bubble map
p_valleys <- ggplot() +
  geom_sf(
    data = world,
    fill = "grey97",
    color = "grey80",
    linewidth = 0.2
  ) +
  geom_sf(
    data = valleys_all %>% filter(!is.na(h2_production_t_year)),
    aes(size = h2_production_t_year, fill = call, color = call),
    shape = 21,
    stroke = 1.4,
    alpha = 0.9
  ) +
  ggrepel::geom_label_repel(
    data = valleys_all,
    aes(label = label_name, geometry = geometry),
    stat = "sf_coordinates",
    size = 4.6,
    fill = "white",
    label.size = 0.2,
    box.padding = 0.45,
    point.padding = 0.5,
    max.overlaps = Inf,
    show.legend = FALSE
  ) +
  scale_fill_manual(
    values = call_palette,
    name = "Funding call",
    guide = guide_legend(
      override.aes = list(
        shape = 21,
        size = 6,
        color = "black",
        stroke = 1.2
      )
    )
  ) +
  scale_color_manual(
    values = call_palette,
    guide = "none"
  ) +
  scale_size_area(
    max_size = 16,
    name = "H2 production\n(t/year)",
    breaks = c(500, 5000, 20000, 50000, 150000),
    labels = scales::comma
  ) +
  coord_sf(
    xlim = common_xlim,
    ylim = common_ylim,
    expand = FALSE
  ) +
  labs(title = "European hydrogen valleys") +
  theme_map

# right: empty corridors map
p_corridors <- ggplot() +
  geom_sf(
    data = world,
    fill = "grey97",
    color = "grey80",
    linewidth = 0.2
  ) +
  coord_sf(
    xlim = common_xlim,
    ylim = common_ylim,
    expand = FALSE
  ) +
  labs(title = "H2 corridors") +
  theme_map +
  theme(
    legend.position = "none"
  )

# combine  panels
edf3 <- p_valleys + p_corridors +
  patchwork::plot_layout(widths = c(1, 1), guides = "collect") &
  theme(legend.position = "right")

options(
  repr.plot.width = 18,
  repr.plot.height = 9,
  repr.plot.res = 600
)

print(edf3)

ggsave(
  "extended-data-figure-1.png",
  edf3,
  width = 18,
  height = 9,
  units = "in",
  dpi = 600,
  limitsize = FALSE
)

In [ ]:
# EXPORT EXTENDED DATA FIGURE 1 SOURCE DATA TO XLSX

# Folder / filename
out_file <- "extended_data_figure_1_source_data.xlsx"


# Prepare all hydrogen-valley data shown in panel a
valleys_source <- valleys_all %>%
  st_transform(4326) %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2],
    name = as.character(name),
    label_name = as.character(label_name),
    call = as.character(call),
    type = as.character(type),
    source_group = case_when(
      name %in% valley_coords$name ~
        "Main hydrogen-valley dataset",
      name %in% additional_valleys$name ~
        "Additional hydrogen-valley dataset",
      TRUE ~
        "Other"
    )
  ) %>%
  st_drop_geometry() %>%
  select(
    name,
    label_name,
    source_group,
    type,
    call,
    h2_production_t_year,
    longitude,
    latitude
  ) %>%
  arrange(
    type,
    call,
    desc(h2_production_t_year),
    name
  )


# Prepare panel-specific hydrogen-valley data
panel_a_valleys <- valleys_source


# Document the intentionally empty corridor panel
panel_b_corridor_status <- tibble(
  panel = "b",
  title = "H2 corridors",
  data_status = "No hydrogen-corridor data plotted",
  plotted_content = paste(
    "Panel contains only the geographic background map",
    "within the shared European map extent"
  )
)


# Prepare original main hydrogen-valley coordinate data
main_valley_coordinates <- valley_coords %>%
  transmute(
    name = as.character(name),
    longitude = lon,
    latitude = lat,
    source_group = "Main hydrogen-valley dataset"
  ) %>%
  arrange(name)


# Prepare original additional hydrogen-valley coordinate data
additional_valley_coordinates <- additional_valleys %>%
  transmute(
    name = as.character(name),
    longitude = lon,
    latitude = lat,
    source_group = "Additional hydrogen-valley dataset"
  ) %>%
  arrange(name)


# Prepare hydrogen-valley metadata
valley_metadata_source <- valley_meta %>%
  mutate(
    name = as.character(name),
    label_name = as.character(label_name),
    call = as.character(call),
    type = as.character(type)
  ) %>%
  select(
    name,
    label_name,
    call,
    type,
    h2_production_t_year
  ) %>%
  arrange(
    type,
    call,
    desc(h2_production_t_year),
    name
  )


# Prepare funding-call summary
funding_call_summary <- valleys_source %>%
  group_by(call) %>%
  summarise(
    n_valleys = n(),
    total_h2_production_t_year = sum(
      h2_production_t_year,
      na.rm = TRUE
    ),
    median_h2_production_t_year = median(
      h2_production_t_year,
      na.rm = TRUE
    ),
    minimum_h2_production_t_year = min(
      h2_production_t_year,
      na.rm = TRUE
    ),
    maximum_h2_production_t_year = max(
      h2_production_t_year,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  mutate(
    call_order = match(
      call,
      names(call_palette)
    )
  ) %>%
  select(
    call_order,
    call,
    n_valleys,
    total_h2_production_t_year,
    median_h2_production_t_year,
    minimum_h2_production_t_year,
    maximum_h2_production_t_year
  ) %>%
  arrange(call_order)


# Prepare hydrogen-valley type summary
valley_type_summary <- valleys_source %>%
  group_by(type) %>%
  summarise(
    n_valleys = n(),
    total_h2_production_t_year = sum(
      h2_production_t_year,
      na.rm = TRUE
    ),
    median_h2_production_t_year = median(
      h2_production_t_year,
      na.rm = TRUE
    ),
    minimum_h2_production_t_year = min(
      h2_production_t_year,
      na.rm = TRUE
    ),
    maximum_h2_production_t_year = max(
      h2_production_t_year,
      na.rm = TRUE
    ),
    .groups = "drop"
  ) %>%
  arrange(type)


# Prepare funding-call palette
funding_call_palette <- tibble(
  call_order = seq_along(call_palette),
  call = names(call_palette),
  colour_hex = unname(call_palette)
)


# Prepare hydrogen-valley type definitions
valley_type_definitions <- tibble(
  type = c(
    "Large-scale H2 Valley",
    "Small-scale H2 Valley"
  ),
  definition = c(
    "Hydrogen valley classified as large-scale in the source data",
    "Hydrogen valley classified as small-scale in the source data"
  )
)


# Prepare bubble-size legend information
production_size_legend <- tibble(
  legend_order = seq_along(
    c(
      500,
      5000,
      20000,
      50000,
      150000
    )
  ),
  h2_production_t_year = c(
    500,
    5000,
    20000,
    50000,
    150000
  ),
  plotted_label = scales::comma(
    c(
      500,
      5000,
      20000,
      50000,
      150000
    )
  )
)


# Prepare map extents
map_extents <- tibble(
  panel = c(
    "Panel a: European hydrogen valleys",
    "Panel b: H2 corridors"
  ),
  xmin = c(
    common_xlim[1],
    common_xlim[1]
  ),
  xmax = c(
    common_xlim[2],
    common_xlim[2]
  ),
  ymin = c(
    common_ylim[1],
    common_ylim[1]
  ),
  ymax = c(
    common_ylim[2],
    common_ylim[2]
  ),
  coordinate_reference_system = "EPSG:4326"
)


# Figure metadata
figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Panel a",
    "Panel b",
    "Coordinate reference system",
    "Geographic background object",
    "Europe-wide longitude range",
    "Europe-wide latitude range",
    "Hydrogen-valley point location",
    "Point fill",
    "Point outline colour",
    "Point size",
    "Point-size scaling",
    "Maximum plotted point size",
    "Point transparency",
    "Point outline width",
    "Point labels",
    "Label-placement method",
    "Funding-call palette",
    "Number of funding calls",
    "Number of hydrogen valleys",
    "Number of large-scale valleys",
    "Number of small-scale valleys",
    "Hydrogen-production unit",
    "Minimum hydrogen production",
    "Maximum hydrogen production",
    "Corridor data status"
  ),
  value = c(
    paste(
      "Extended Data Figure 1:",
      "European hydrogen valleys and H2 corridors"
    ),
    paste(
      "European map showing hydrogen valleys classified",
      "by funding call and scaled by annual hydrogen production,",
      "alongside an empty hydrogen-corridor map"
    ),
    paste(
      "European hydrogen valleys coloured by funding call",
      "and sized by annual hydrogen production"
    ),
    paste(
      "European H2 corridor map containing only",
      "the geographic background"
    ),
    "EPSG:4326",
    "world",
    paste(
      common_xlim,
      collapse = " to "
    ),
    paste(
      common_ylim,
      collapse = " to "
    ),
    "Project longitude and latitude",
    "Funding call",
    "Funding call",
    "Annual hydrogen production",
    "scale_size_area",
    "16",
    "0.9",
    "1.4",
    "Hydrogen-valley label name",
    "ggrepel::geom_label_repel",
    "NPG NRC palette with the fifth and ninth colours swapped",
    as.character(length(call_palette)),
    as.character(nrow(valleys_source)),
    as.character(
      sum(
        valleys_source$type ==
          "Large-scale H2 Valley",
        na.rm = TRUE
      )
    ),
    as.character(
      sum(
        valleys_source$type ==
          "Small-scale H2 Valley",
        na.rm = TRUE
      )
    ),
    "Tonnes of hydrogen per year",
    as.character(
      min(
        valleys_source$h2_production_t_year,
        na.rm = TRUE
      )
    ),
    as.character(
      max(
        valleys_source$h2_production_t_year,
        na.rm = TRUE
      )
    ),
    paste(
      "No hydrogen-corridor geometries or points",
      "were plotted in panel b"
    )
  )
)


# Panel descriptions
panel_metadata <- tibble(
  panel = c(
    "a",
    "b"
  ),
  title = c(
    "European hydrogen valleys",
    "H2 corridors"
  ),
  geographic_extent = c(
    "Europe",
    "Europe"
  ),
  source_object = c(
    "valleys_all",
    "world"
  ),
  plotted_data = c(
    paste(
      "Hydrogen-valley locations, funding calls,",
      "annual hydrogen production and project labels"
    ),
    "No corridor data; geographic background only"
  ),
  point_fill = c(
    "Funding call",
    NA_character_
  ),
  point_size = c(
    "Annual hydrogen production in tonnes per year",
    NA_character_
  ),
  additional_annotation = c(
    "Hydrogen-valley project labels",
    NA_character_
  )
)


# Source sheets
source_sheets <- list(
  "README" = figure_metadata,

  "Panel_descriptions" = panel_metadata,

  "Panel_a_valleys" = panel_a_valleys,

  "Panel_b_corridor_status" = panel_b_corridor_status,

  "All_valleys" = valleys_source,

  "Main_valley_coordinates" = main_valley_coordinates,

  "Additional_valleys" = additional_valley_coordinates,

  "Valley_metadata" = valley_metadata_source,

  "Funding_call_summary" = funding_call_summary,

  "Valley_type_summary" = valley_type_summary,

  "Funding_call_palette" = funding_call_palette,

  "Valley_type_definitions" = valley_type_definitions,

  "Production_size_legend" = production_size_legend,

  "Map_extents" = map_extents
)


# Write workbook
wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  sheet_data <- source_sheets[[sheet_name]]

  addWorksheet(
    wb,
    sheetName = sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = sheet_data
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  if (ncol(sheet_data) > 0) {
    addFilter(
      wb,
      sheet = sheet_name,
      row = 1,
      cols = seq_len(ncol(sheet_data))
    )

    setColWidths(
      wb,
      sheet = sheet_name,
      cols = seq_len(ncol(sheet_data)),
      widths = "auto"
    )
  }
})

saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)

In [ ]:
# EXTENDED DATA FIGURE: DISTRIBUTION OF SPILLOVER POTENTIAL BY REGION (EXCLUDING HEAT)
# ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# centrality scale transformations
power_04_trans <- scales::trans_new(
  name = "power_04",
  transform = function(x) x^0.4,
  inverse = function(x) x^(1 / 0.4)
)

sqrt_trans <- scales::trans_new(
  name = "sqrt_custom",
  transform = function(x) sqrt(x),
  inverse = function(x) x^2
)

identity_trans <- scales::trans_new(
  name = "identity_custom",
  transform = function(x) x,
  inverse = function(x) x
)

# use identity scale for percentile values
transformation <- identity_trans

# plotting settings
excluded_sector <- "Heat"

pop_max_cutoff_zoom1 <- 1000000
pop_max_cutoff_zoom2 <- 250000
highlight <- 50
common_xlim <- c(-15, 45)
common_ylim <- c(30, 75)
mean_lat_global <- mean(common_ylim)

# helper for longitude scaling in zoom windows
deg2rad <- function(d) d * pi / 180

# zoom box 1: north rhine-westphalia
zoom1_center <- c(lon = 7.0, lat = 51.5)
zoom1_height <- 2
zoom1_width <- zoom1_height *
  (diff(common_xlim) * cos(deg2rad(mean_lat_global))) /
  (diff(common_ylim) * cos(deg2rad(zoom1_center["lat"])))

zoom1_xlim <- c(
  zoom1_center["lon"] - zoom1_width / 2,
  zoom1_center["lon"] + zoom1_width / 2
)

zoom1_ylim <- c(
  zoom1_center["lat"] - zoom1_height / 2,
  zoom1_center["lat"] + zoom1_height / 2
)

# zoom box 2: eastern austria / moravia
zoom2_center <- c(lon = 16.5, lat = 48.2)
zoom2_height <- 3
zoom2_width <- zoom2_height *
  (diff(common_xlim) * cos(deg2rad(mean_lat_global))) /
  (diff(common_ylim) * cos(deg2rad(zoom2_center["lat"])))

zoom2_xlim <- c(
  zoom2_center["lon"] - zoom2_width / 2,
  zoom2_center["lon"] + zoom2_width / 2
)

zoom2_ylim <- c(
  zoom2_center["lat"] - zoom2_height / 2,
  zoom2_center["lat"] + zoom2_height / 2
)

# prepare long centrality data and convert scores to within-metric percentiles
offtakers_long <- offtakers %>%
  st_transform(4326) %>%
  filter(sector != excluded_sector) %>%
  select(
    plant_id,
    installation_name,
    geometry,
    sector,
    degree,
    betweenness
  ) %>%
  pivot_longer(
    cols = c(degree, betweenness),
    names_to = "centrality_type",
    values_to = "centrality_score"
  ) %>%
  group_by(centrality_type) %>%
  mutate(
    centrality_pct = percent_rank(centrality_score)
  ) %>%
  ungroup()

# helper to build bbox polygons
make_bbox_polygon <- function(xmin, xmax, ymin, ymax, crs = 4326) {
  coords <- matrix(
    c(
      xmin, ymin,
      xmin, ymax,
      xmax, ymax,
      xmax, ymin,
      xmin, ymin
    ),
    ncol = 2,
    byrow = TRUE
  )

  st_sfc(st_polygon(list(coords)), crs = crs)
}

bbox_zoom1 <- make_bbox_polygon(
  zoom1_xlim[1], zoom1_xlim[2],
  zoom1_ylim[1], zoom1_ylim[2]
)

bbox_zoom2 <- make_bbox_polygon(
  zoom2_xlim[1], zoom2_xlim[2],
  zoom2_ylim[1], zoom2_ylim[2]
)

# helper to extract top n sites within a zoom area
get_topN_zoom <- function(data, metric, bbox, n = highlight) {
  data %>%
    filter(centrality_type == metric) %>%
    st_intersection(bbox) %>%
    arrange(desc(centrality_score)) %>%
    slice_head(n = n)
}

# sectors represented among highlighted top sites
active_deg1 <- get_topN_zoom(
  offtakers_long,
  "degree",
  bbox_zoom1
) %>%
  st_drop_geometry() %>%
  pull(sector) %>%
  unique()

active_bet2 <- get_topN_zoom(
  offtakers_long,
  "betweenness",
  bbox_zoom2
) %>%
  st_drop_geometry() %>%
  pull(sector) %>%
  unique()

# sector palette
nrc_base10 <- pal_npg("nrc")(10)

extra_colors <- c(
  "#7A7A7A",
  "#8C564B"
)

nrc_colors <- c(nrc_base10, extra_colors)

sector_levels <- sort(unique(offtakers_long$sector))

special_colors <- c(
  "Other" = "#808080",
  "Iron & steel" = nrc_colors[1],
  "Non-metallic minerals" = nrc_colors[5]
)

special_colors <- special_colors[names(special_colors) %in% sector_levels]

reserved_colors <- unique(unname(special_colors))

regular_sector_levels <- setdiff(sector_levels, names(special_colors))

regular_colors <- setdiff(nrc_colors, reserved_colors)

if (length(regular_sector_levels) > length(regular_colors)) {
  stop("Not enough non-reserved colours for all regular sectors.")
}

sector_palette <- c(
  setNames(regular_colors[seq_along(regular_sector_levels)], regular_sector_levels),
  special_colors
)

sector_palette <- sector_palette[sector_levels]

offtakers_long$sector <- factor(
  offtakers_long$sector,
  levels = sector_levels
)

# city labels for zoom panels
cities <- ne_download(
  scale = 10,
  type = "populated_places",
  category = "cultural",
  returnclass = "sf"
)

cities_zoom1 <- st_intersection(cities, bbox_zoom1) %>%
  filter(POP_MAX > pop_max_cutoff_zoom1)

cities_zoom2 <- st_intersection(cities, bbox_zoom2) %>%
  filter(POP_MAX > pop_max_cutoff_zoom2)

# sectors that receive ring highlighting in the legend
ring_sectors_global <- union(active_deg1, active_bet2)

legend_df <- data.frame(
  sector = sector_levels,
  is_ring = sector_levels %in% ring_sectors_global,
  x = 1000,
  y = 1000
)

# europe-wide centrality map
plot_centrality_simple <- function(
  data,
  metric,
  title,
  valleys,
  valleys_small,
  show_legend = FALSE,
  bbox = NULL
) {
  ggplot() +
    geom_sf(
      data = world,
      fill = "grey97",
      color = "grey85",
      size = 0.2
    ) +
    geom_sf(
      data = filter(data, centrality_type == metric),
      aes(fill = centrality_pct),
      shape = 21,
      size = 0.7,
      stroke = 0,
      alpha = 0.7
    ) +
    geom_sf(
      data = valleys,
      aes(shape = type, color = type),
      size = 3.5,
      stroke = 1.2,
      fill = NA
    ) +
    geom_sf(
      data = valleys_small,
      aes(shape = type, color = type),
      size = 3.0,
      stroke = 1.2,
      fill = NA
    ) +
{
  if (!is.null(bbox)) list(
    geom_sf(
      data = bbox,
      fill = NA,
      color = "white",
      linewidth = 2.8
    ),
    geom_sf(
      data = bbox,
      fill = NA,
      color = "black",
      linewidth = 1.2
    )
  )
} +
    scale_fill_viridis_c(
      option = "mako",
      direction = -1,
      limits = c(0, 1),
      trans = transformation,
      name = "Centrality\n(percentile)"
    ) +
    scale_shape_manual(
      values = c(
        "Large-scale H2 Valley" = 4,
        "Small-scale H2 Valley" = 1
      ),
      name = "Hydrogen valleys"
    ) +
    scale_color_manual(
      values = c(
        "Large-scale H2 Valley" = "maroon",
        "Small-scale H2 Valley" = "maroon"
      ),
      name = "Hydrogen valleys"
    ) +
    coord_sf(
      xlim = common_xlim,
      ylim = common_ylim,
      expand = FALSE
    ) +
    labs(title = title) +
    theme_classic(base_size = 16) +
    theme(
      plot.title = element_text(hjust = 0),
      legend.position = if (show_legend) "right" else "none",
      panel.border = element_rect(
        color = "black",
        fill = NA,
        linewidth = 0.6
      )
    )
}

# zoomed map with top sites ringed by sector
plot_centrality_sector <- function(
  data,
  metric,
  title,
  valleys,
  valleys_small,
  show_legend = FALSE,
  zoom = FALSE,
  zoom_xlim = NULL,
  zoom_ylim = NULL,
  cities_zoom = NULL
) {
  data_sub <- data %>%
    filter(centrality_type == metric)

  topN <- get_topN_zoom(
    data,
    metric,
    make_bbox_polygon(
      zoom_xlim[1],
      zoom_xlim[2],
      zoom_ylim[1],
      zoom_ylim[2]
    )
  )

  data_sub <- data_sub %>%
    mutate(is_topN = plant_id %in% topN$plant_id)

  ggplot() +
    geom_sf(
      data = world,
      fill = "grey97",
      color = "grey85",
      size = 0.2
    ) +
    geom_sf(
      data = data_sub %>% filter(!is_topN),
      aes(fill = centrality_pct),
      shape = 21,
      size = 2,
      stroke = 0,
      alpha = 0.4
    ) +
    geom_sf(
      data = topN,
      aes(fill = centrality_pct),
      shape = 21,
      size = 7,
      stroke = 0,
      alpha = 0.3
    ) +
    geom_sf(
      data = topN,
      aes(color = sector),
      shape = 21,
      size = 7,
      stroke = 2,
      fill = NA
    ) +
    # invisible points used to force full legend entries
    geom_point(
      data = legend_df,
      aes(x = x, y = y, color = sector),
      shape = 16,
      size = 3,
      inherit.aes = FALSE
    ) +
    geom_point(
      data = legend_df %>% filter(is_ring),
      aes(x = x, y = y, color = sector),
      shape = 21,
      fill = NA,
      size = 7,
      stroke = 2,
      inherit.aes = FALSE
    ) +
    {
      if (zoom) geom_sf(
        data = cities_zoom,
        color = "black",
        size = 1.2
      )
    } +
    {
      if (zoom) geom_label_repel(
        data = cities_zoom,
        aes(label = NAME, geometry = geometry),
        stat = "sf_coordinates",
        size = 4,
        fill = "white",
        box.padding = 0.4,
        label.size = 0.2
      )
    } +
    scale_fill_viridis_c(
      option = "mako",
      direction = -1,
      limits = c(0, 1),
      trans = transformation,
      name = "Centrality\n(percentile)"
    ) +
    scale_color_manual(
      name = "Top 50 offtakers / Sector",
      values = sector_palette,
      breaks = sector_levels,
      drop = FALSE,
      guide = if (show_legend) "legend" else "none"
    ) +
    coord_sf(
      xlim = zoom_xlim,
      ylim = zoom_ylim,
      expand = FALSE
    ) +
    labs(title = title) +
    theme_classic(base_size = 16) +
    theme(
      plot.title = element_text(hjust = 0),
      legend.position = if (show_legend) "right" else "none",
      panel.border = element_rect(
        color = "black",
        fill = NA,
        linewidth = 0.6
      )
    )
}

# panel a
p1 <- plot_centrality_simple(
  offtakers_long,
  "degree",
  "H₂ Valleys | Degree Centrality",
  valleys_sf,
  additional_valleys_sf,
  show_legend = FALSE,
  bbox = bbox_zoom1
)

# panel b
p2 <- plot_centrality_simple(
  offtakers_long,
  "betweenness",
  "H₂ Corridors | Betweenness Centrality",
  empty_valleys,
  empty_additional,
  show_legend = TRUE,
  bbox = bbox_zoom2
)

# panel c
p3 <- plot_centrality_sector(
  offtakers_long,
  "degree",
  "North Rhine-Westphalia",
  valleys_sf,
  additional_valleys_sf,
  zoom = TRUE,
  show_legend = FALSE,
  zoom_xlim = zoom1_xlim,
  zoom_ylim = zoom1_ylim,
  cities_zoom = cities_zoom1
)

# panel d
p4 <- plot_centrality_sector(
  offtakers_long,
  "betweenness",
  "Eastern Austria / Moravia",
  valleys_sf,
  additional_valleys_sf,
  zoom = TRUE,
  show_legend = TRUE,
  zoom_xlim = zoom2_xlim,
  zoom_ylim = zoom2_ylim,
  cities_zoom = cities_zoom2
)

# combine panels
edf3 <- ((p1 | p2) / (p3 | p4)) +
  plot_annotation(
    tag_levels = "a",
    theme = theme(
      plot.tag = element_text(
        face = "bold",
        size = 14
      )
    )
  ) +
  plot_layout(
    guides = "collect",
    widths = c(1, 1),
    heights = c(1, 1)
  ) &
  theme(
    legend.position = "right",
    panel.spacing = unit(0, "cm"),
    axis.text = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank(),
    plot.margin = margin(0, 0, 0, 0)
  )

# display settings
options(
  repr.plot.width = 16,
  repr.plot.height = 12,
  repr.plot.res = 600
)

print(edf3)

# export figure
ggsave(
  "extended-data-figure-3.png",
  edf3,
  width = 16,
  height = 12,
  units = "in",
  dpi = 600,
  limitsize = FALSE
)

In [ ]:
# EXPORT EXTENDED DATA FIGURE 3 SOURCE DATA TO XLSX

# Folder / filename
out_file <- "extended_data_figure_3_source_data.xlsx"


# Prepare installation-level map data
offtakers_source <- offtakers_long %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2],
    centrality_type = as.character(centrality_type),
    sector = as.character(sector)
  ) %>%
  st_drop_geometry() %>%
  select(
    plant_id,
    installation_name,
    sector,
    centrality_type,
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  ) %>%
  arrange(
    centrality_type,
    desc(centrality_score),
    plant_id
  )


# Prepare panel-specific centrality data
panel_a_degree <- offtakers_source %>%
  filter(centrality_type == "degree")

panel_b_betweenness <- offtakers_source %>%
  filter(centrality_type == "betweenness")


# Reproduce highlighted top-50 sites in zoom panels
panel_c_top50_degree <- get_topN_zoom(
  data = offtakers_long,
  metric = "degree",
  bbox = bbox_zoom1,
  n = highlight
) %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    rank_within_zoom = row_number(),
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  )

panel_d_top50_betweenness <- get_topN_zoom(
  data = offtakers_long,
  metric = "betweenness",
  bbox = bbox_zoom2,
  n = highlight
) %>%
  mutate(
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    rank_within_zoom = row_number(),
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    longitude,
    latitude
  )


# Prepare all sites shown inside each zoom panel
panel_c_all_sites <- offtakers_long %>%
  filter(centrality_type == "degree") %>%
  st_filter(bbox_zoom1, .predicate = st_intersects) %>%
  mutate(
    is_top50 = plant_id %in% panel_c_top50_degree$plant_id,
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    is_top50,
    longitude,
    latitude
  ) %>%
  arrange(
    desc(is_top50),
    desc(centrality_score),
    plant_id
  )

panel_d_all_sites <- offtakers_long %>%
  filter(centrality_type == "betweenness") %>%
  st_filter(bbox_zoom2, .predicate = st_intersects) %>%
  mutate(
    is_top50 = plant_id %in% panel_d_top50_betweenness$plant_id,
    longitude = st_coordinates(.)[, 1],
    latitude = st_coordinates(.)[, 2]
  ) %>%
  st_drop_geometry() %>%
  transmute(
    plant_id,
    installation_name,
    sector = as.character(sector),
    centrality_type = as.character(centrality_type),
    centrality_score,
    centrality_percentile = centrality_pct,
    is_top50,
    longitude,
    latitude
  ) %>%
  arrange(
    desc(is_top50),
    desc(centrality_score),
    plant_id
  )


# Prepare hydrogen-valley point data
prepare_valley_source <- function(x, source_group) {

  if (nrow(x) == 0) {
    return(
      tibble(
        source_group = character(),
        name = character(),
        type = character(),
        longitude = numeric(),
        latitude = numeric()
      )
    )
  }

  x_out <- x %>%
    st_transform(4326) %>%
    mutate(
      longitude = st_coordinates(.)[, 1],
      latitude = st_coordinates(.)[, 2],
      source_group = source_group
    )

  if (!"name" %in% names(x_out)) {
    x_out <- x_out %>%
      mutate(name = NA_character_)
  }

  x_out %>%
    st_drop_geometry() %>%
    transmute(
      source_group,
      name = as.character(name),
      type = as.character(type),
      longitude,
      latitude
    )
}

hydrogen_valleys_source <- bind_rows(
  prepare_valley_source(
    valleys_sf,
    "Main hydrogen-valley dataset"
  ),
  prepare_valley_source(
    additional_valleys_sf,
    "Additional hydrogen-valley dataset"
  )
)


# Prepare city-label data
prepare_city_source <- function(x, panel_name) {

  if (nrow(x) == 0) {
    return(
      tibble(
        panel = character(),
        city = character(),
        population_max = numeric(),
        longitude = numeric(),
        latitude = numeric()
      )
    )
  }

  x %>%
    st_transform(4326) %>%
    mutate(
      longitude = st_coordinates(.)[, 1],
      latitude = st_coordinates(.)[, 2]
    ) %>%
    st_drop_geometry() %>%
    transmute(
      panel = panel_name,
      city = NAME,
      population_max = POP_MAX,
      longitude,
      latitude
    ) %>%
    arrange(desc(population_max), city)
}

city_labels_source <- bind_rows(
  prepare_city_source(
    cities_zoom1,
    "Panel c: North Rhine-Westphalia"
  ),
  prepare_city_source(
    cities_zoom2,
    "Panel d: Eastern Austria / Moravia"
  )
)


# Prepare map extents and zoom-box coordinates
map_extents <- tibble(
  panel = c(
    "Panels a-b: Europe",
    "Panel c: North Rhine-Westphalia",
    "Panel d: Eastern Austria / Moravia"
  ),
  xmin = c(
    common_xlim[1],
    zoom1_xlim[1],
    zoom2_xlim[1]
  ),
  xmax = c(
    common_xlim[2],
    zoom1_xlim[2],
    zoom2_xlim[2]
  ),
  ymin = c(
    common_ylim[1],
    zoom1_ylim[1],
    zoom2_ylim[1]
  ),
  ymax = c(
    common_ylim[2],
    zoom1_ylim[2],
    zoom2_ylim[2]
  ),
  coordinate_reference_system = "EPSG:4326"
)

zoom_box_vertices <- bind_rows(
  tibble(
    panel = "Panel a zoom box",
    vertex_order = 1:5,
    longitude = c(
      zoom1_xlim[1],
      zoom1_xlim[1],
      zoom1_xlim[2],
      zoom1_xlim[2],
      zoom1_xlim[1]
    ),
    latitude = c(
      zoom1_ylim[1],
      zoom1_ylim[2],
      zoom1_ylim[2],
      zoom1_ylim[1],
      zoom1_ylim[1]
    )
  ),
  tibble(
    panel = "Panel b zoom box",
    vertex_order = 1:5,
    longitude = c(
      zoom2_xlim[1],
      zoom2_xlim[1],
      zoom2_xlim[2],
      zoom2_xlim[2],
      zoom2_xlim[1]
    ),
    latitude = c(
      zoom2_ylim[1],
      zoom2_ylim[2],
      zoom2_ylim[2],
      zoom2_ylim[1],
      zoom2_ylim[1]
    )
  )
)


# Prepare sector palette and legend information
sector_palette_source <- tibble(
  sector = names(sector_palette),
  colour_hex = unname(sector_palette),
  represented_among_highlighted_sites =
    names(sector_palette) %in% ring_sectors_global
)


# Figure metadata
figure_metadata <- tibble(
  item = c(
    "Figure",
    "Description",
    "Coordinate reference system",
    "Excluded sector",
    "Centrality metrics",
    "Centrality display unit",
    "Centrality percentile calculation",
    "Centrality colour scale",
    "Centrality scale transformation",
    "Alternative scale objects defined but not used",
    "Number of highlighted sites per zoom panel",
    "City population threshold",
    "Europe-wide longitude range",
    "Europe-wide latitude range",
    "Panel c centre longitude",
    "Panel c centre latitude",
    "Panel d centre longitude",
    "Panel d centre latitude",
    "Hydrogen valleys shown in panel a",
    "Hydrogen valleys shown in panel b",
    "Legend ring definition"
  ),
  value = c(
    "Extended Data Figure 3: Distribution of spillover potential by region (excluding heat)",
    paste(
      "Europe-wide degree and betweenness centrality maps,",
      "with zoom panels showing the 50 highest-ranked offtakers",
      "within North Rhine-Westphalia and Eastern Austria / Moravia"
    ),
    "EPSG:4326",
    excluded_sector,
    "Degree centrality; Betweenness centrality",
    "Within-metric percentile, ranging from 0 to 1",
    paste(
      "percent_rank applied separately within degree and",
      "betweenness centrality"
    ),
    "Viridis mako, reversed",
    "Identity",
    "power_04_trans; sqrt_trans",
    as.character(highlight),
    as.character(pop_max_cutoff),
    paste(common_xlim, collapse = " to "),
    paste(common_ylim, collapse = " to "),
    as.character(zoom1_center["lon"]),
    as.character(zoom1_center["lat"]),
    as.character(zoom2_center["lon"]),
    as.character(zoom2_center["lat"]),
    "Main and additional hydrogen-valley datasets",
    "None; empty valley objects used",
    "Sectors represented among highlighted top sites in either zoom panel"
  )
)


# Panel descriptions
panel_metadata <- tibble(
  panel = c(
    "a",
    "b",
    "c",
    "d"
  ),
  title = c(
    "H2 Valleys | Degree Centrality",
    "H2 Corridors | Betweenness Centrality",
    "North Rhine-Westphalia",
    "Eastern Austria / Moravia"
  ),
  centrality_metric = c(
    "degree",
    "betweenness",
    "degree",
    "betweenness"
  ),
  geographic_extent = c(
    "Europe",
    "Europe",
    "North Rhine-Westphalia zoom window",
    "Eastern Austria / Moravia zoom window"
  ),
  highlighted_sites = c(
    NA_integer_,
    NA_integer_,
    highlight,
    highlight
  ),
  hydrogen_valleys_plotted = c(
    "Yes",
    "No",
    "No",
    "No"
  )
)


# Source sheets
source_sheets <- list(
  "README" = figure_metadata,

  "Panel_descriptions" = panel_metadata,

  "All_offtakers" = offtakers_source,

  "Panel_a_degree" = panel_a_degree,

  "Panel_b_betweenness" = panel_b_betweenness,

  "Panel_c_all_sites" = panel_c_all_sites,

  "Panel_c_top50" = panel_c_top50_degree,

  "Panel_d_all_sites" = panel_d_all_sites,

  "Panel_d_top50" = panel_d_top50_betweenness,

  "Hydrogen_valleys" = hydrogen_valleys_source,

  "City_labels" = city_labels_source,

  "Map_extents" = map_extents,

  "Zoom_box_vertices" = zoom_box_vertices,

  "Sector_palette" = sector_palette_source
)


# Write workbook
wb <- createWorkbook()

walk(names(source_sheets), function(sheet_name) {

  sheet_data <- source_sheets[[sheet_name]]

  addWorksheet(
    wb,
    sheetName = sheet_name
  )

  writeData(
    wb,
    sheet = sheet_name,
    x = sheet_data
  )

  freezePane(
    wb,
    sheet = sheet_name,
    firstRow = TRUE
  )

  if (ncol(sheet_data) > 0) {
    addFilter(
      wb,
      sheet = sheet_name,
      row = 1,
      cols = seq_len(ncol(sheet_data))
    )

    setColWidths(
      wb,
      sheet = sheet_name,
      cols = seq_len(ncol(sheet_data)),
      widths = "auto"
    )
  }
})

saveWorkbook(
  wb,
  out_file,
  overwrite = TRUE
)